<a href="https://colab.research.google.com/github/ileeee-sh/SeSAC_29CM_Project/blob/main/29cm_Playstore_comment_SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3계층 감성사전 기반 감성분석
**파이프라인**: KNU(1.5) + 디자인(1.2) + 이모지(0.35) → TF-IDF 가중 평균 → polarity (-1.0 ~ +1.0)

In [ ]:
!pip install openpyxl scikit-learn --quiet

In [ ]:
import pandas as pd
import numpy as np
import requests, re, ast
from sklearn.feature_extraction.text import TfidfVectorizer
from google.colab import files

**이모지 감성사전(직접제작)**

**1. Positive**

🔥 최고 긍정 (1.0) - 18개

    '👍': 1.0, '❤️': 1.0, '💖': 1.0, '💕': 1.0, '💗': 1.0, '💓': 1.0,
    '🤍': 1.0, '🧡': 1.0, '💛': 1.0, '💚': 1.0, '💙': 1.0, '💜': 1.0,
    '🖤': 1.0, '💝': 1.0, '💞': 1.0, '💟': 1.0, '💯': 1.0,

⭐ 강한 긍정 (0.80~0.95) - 13개

    '😍': 0.95, '⭐': 0.92, '✨': 0.88, '🥰': 0.85, '😁': 0.83,
    '😊': 0.82, '😚': 0.82, '😂': 0.80, '🙌': 0.78, '😘': 0.75,
    '✅': 0.75, '🚀': 0.75,

😌 중간 긍정 (0.60~0.78) - 8개

     '😌': 0.75, '🎉': 0.70, '😸': 0.7, '🐵': 0.6,

**2. Negative**

😡 최고 부정 (-1.0) - 4개

    '👎': -1.0, '💔': -1.0, '😡': -0.95,

⚠️ 강한 부정 (-0.85~-0.90) - 8개

    '🤢': -0.9, '🚨': -0.90, '😭': -0.90, '❗': -0.85, '🛑': -0.85,
    '😱': -0.85, '🤬': -0.85, '😤': -0.8,

😓 중간 부정 (-0.60~-0.75) - 8개

    '😣': -0.80, '😰': -0.75, '😩': -0.75, '😓': -0.70, '❌': -0.75,
    '⏳': -0.60, '🔄': -0.45, '⌛': -0.50,

🙄 약한 부정 (-0.50~-0.65) - 4개

    '🙄': -0.65, '😒': -0.65, '😥': -0.65,

**3.Neutral**

⚙️ 중립/사용성 - 8개

    '📱': 0.0, '⚙️': 0.0, '🔧': -0.20, '📶': 0.15, '🔋': 0.10,
    '🔔': 0.10, '🔕': -0.20, '📴': -0.35

In [ ]:
emoji_lexicon = {
    # 최고 긍정 (1.0)
    '👍': 1.0, '❤️': 1.0, '💖': 1.0, '💕': 1.0, '💗': 1.0, '💓': 1.0,
    '🤍': 1.0, '🧡': 1.0, '💛': 1.0, '💚': 1.0, '💙': 1.0, '💜': 1.0,
    '🖤': 1.0, '💝': 1.0, '💞': 1.0, '💟': 1.0, '💯': 1.0,
    # 강한 긍정 (0.75~0.95)
    '😍': 0.95, '⭐': 0.92, '✨': 0.88, '🥰': 0.85, '😁': 0.83,
    '😊': 0.82, '😚': 0.82, '😂': 0.80, '🙌': 0.78, '😘': 0.75,
    '✅': 0.75, '🚀': 0.75,
    # 중간 긍정 (0.60~0.75)
    '😌': 0.75, '🎉': 0.70, '😸': 0.70, '🐵': 0.60,
    # 최고 부정
    '👎': -1.0, '💔': -1.0, '😡': -0.95,
    # 강한 부정
    '🤢': -0.90, '🚨': -0.90, '😭': -0.90, '❗': -0.85, '🛑': -0.85,
    '😱': -0.85, '🤬': -0.85, '😤': -0.80,
    # 중간 부정
    '😣': -0.80, '😰': -0.75, '😩': -0.75, '😓': -0.70, '❌': -0.75,
    '⏳': -0.60, '🔄': -0.45, '⌛': -0.50,
    # 약한 부정
    '🙄': -0.65, '😒': -0.65, '😥': -0.65,
    # 중립/사용성
    '📱': 0.0, '⚙️': 0.0, '🔧': -0.20, '📶': 0.15, '🔋': 0.10,
    '🔔': 0.10, '🔕': -0.20, '📴': -0.35
}
print(f'이모지 감성사전: {len(emoji_lexicon)}개')

In [ ]:
import requests
knu_url = "https://raw.githubusercontent.com/park1200656/KnuSentiLex/master/SentiWord_Dict.txt"

response = requests.get(knu_url)
response.encoding = 'utf-8'

rows = []
for line in response.text.strip().split('\n'):
    parts = line.strip().split('\t')
    if len(parts) == 2:
        try:
            rows.append({'word': parts[0].strip(), 'polarity': float(parts[1].strip())})
        except ValueError:
            pass

knu_df = pd.DataFrame(rows)

# polarity: 2 ~ -2 → -1.0~1.0 (Normalization)
knu_df['score'] = knu_df['polarity'] / 2.0
knu_lexicon = dict(zip(knu_df['word'], knu_df['score']))

print(f'KNU 감성사전: {len(knu_lexicon):,}개')
print('샘플:', list(knu_lexicon.items())[:5])

In [ ]:
# 리뷰 특화 커스텀 사전 (수동 추가)
custom_lexicon = {
    # 긍정 구어체
    '좋아요': 1.0, '좋아': 0.9, '좋네요': 0.9, '좋습니다': 1.0,
    '굿': 0.9, '굳': 0.9, '굿굿': 1.0, '최고예요': 1.0, '최고': 0.9,
    '이뻐요': 1.0, '이뻐': 0.9, '이쁘다': 0.9, '이쁘네요': 0.9, '이쁜': 0.8,
    '예뻐요': 1.0, '예뻐': 0.9, '예쁘다': 0.9, '예쁜': 0.8,
    '맘에들어요': 1.0, '맘에들어': 0.9, '마음에들어요': 1.0,
    '완벽해요': 1.0, '만족해요': 0.9, '만족합니다': 0.9,
    '추천해요': 0.9, '추천합니다': 0.9, '강추': 1.0,
    '감사해요': 0.8, '감사합니다': 0.8,
    '편해요': 0.8, '편하다': 0.8, '편합니다': 0.8,
    '빨라요': 0.7, '빠르다': 0.7,
    '친절해요': 0.8, '친절합니다': 0.8,
    '깔끔해요': 0.8, '깔끔하다': 0.8,
    '행복해요': 0.9, '기뻐요': 0.9,
    "최애": 1.0, "덕분에": 0.8, "갬성": 0.7, "갠춘": 0.6, "괜츈": 0.6,
    "할인": 0.65, "쿠폰": 0.65,


    # 부정 구어체
    "별로예요": -0.8, '별로에요': -0.8, '별로': -0.7,
    '실망이에요': -0.9, '실망했어요': -0.9, '실망': -0.8,
    "그따구로": -1.0, "글러먹은": -1.0, "무책임한": -1.0,
    "먹튀앱": -1.0, "성가셔": -0.7,
    '불편해요': -0.8, '불편하다': -0.8,
    '느려요': -0.7, '느리다': -0.7,
    '비싸요': -0.6, '비싸다': -0.6,
    '최악이에요': -1.0, '최악': -1.0,
    '아쉬워요': -0.6, '아쉽다': -0.6,
    '별로네요': -0.8, '그냥그래요': -0.4,
    "다만": -0.2, "결국": -0.2, "제발": -0.5,
    "X같이": -1.0, "렉": -0.6, "안": -0.75, "버벅": -0.6, "버벅거": -0.6, "잣": -1.0,
    "개떡": -1.0, "오류": -0.8, "에러": -0.8, "에휴": -0.9, "불량": -0.8,
    "삭제": -0.8, "벌레": -0.7, "개그지": -0.7, "싫어": -0.8, "꺼져": -0.7, "강제": -0.7, "쓰레기": -0.8
}

# KNU에 병합 (커스텀이 우선)
knu_lexicon.update(custom_lexicon)
print(f'커스텀 사전 {len(custom_lexicon)}개 추가 → 총 {len(knu_lexicon):,}개')

In [ ]:
uploaded = files.upload()
design_file = [f for f in uploaded.keys() if 'Design' in f or 'design' in f][0]

design_df = pd.read_excel(design_file, usecols=['감성어', '감성', '감성매력', '구매유도', '경험가치'])
design_df.columns = ['word', 'sentiment', 'attract', 'purchase', 'experience']
design_df = design_df.dropna(subset=['word'])

# 한글 부분만 추출: '편안함 (Comfort)' → '편안함'
design_df['word_ko'] = design_df['word'].apply(
    lambda x: re.sub(r'\s*\(.*?\)', '', str(x)).strip()
)

# 점수 합산 (감성매력+구매유도+경험가치, 범위 0~15) → -1.0~1.0 정규화
# 긍정: (sum/15)*1.0, 부정: -(sum/15)*1.0, 중립: 0
def calc_design_score(row):
    total = sum(v for v in [row['attract'], row['purchase'], row['experience']]
                if isinstance(v, (int, float)))
    normalized = total / 15.0  # 0~1
    sentiment = str(row['sentiment']).strip()
    if '긍정' in sentiment:
        return normalized
    elif '부정' in sentiment:
        return -normalized
    else:  # 중립
        return 0.0

design_df['score'] = design_df.apply(calc_design_score, axis=1)

# 같은 단어가 여러 분야에 등장 → 평균값 사용
design_lexicon = design_df.groupby('word_ko')['score'].mean().to_dict()

print(f'디자인 감성사전: {len(design_lexicon)}개 고유 단어')
print('샘플:', list(design_lexicon.items())[:5])

In [ ]:
uploaded2 = files.upload()
review_file = list(uploaded2.keys())[0]

df = pd.read_csv(review_file)
print(f'파일 로드: {df.shape[0]}행')
print('컬럼 목록:', list(df.columns))
df.head(3)

In [ ]:
token_col = df.columns[7]
print(f'토큰 컬럼명: "{token_col}"')
print('샘플:', df[token_col].iloc[0])

In [ ]:
emoji_col = df.columns[4]
print(f'이모지 컬럼명: "{emoji_col}"')
print('샘플:', df[emoji_col].iloc[0])

In [ ]:
#TF-IDF
def parse_list_col(val):
    if isinstance(val, list):
        return val
    if pd.isna(val) or val == '' or val == '[]':
        return []
    try:
        return ast.literal_eval(str(val))
    except:
        return str(val).strip("[]").replace("'", "").split(", ")

df['tokens_list']  = df[token_col].apply(parse_list_col)
df['emojis_list']  = df[emoji_col].apply(parse_list_col) if emoji_col in df.columns else [[] for _ in range(len(df))]

# TF-IDF 계산용: 토큰 리스트를 공백 합친 문자열로
corpus = df['tokens_list'].apply(lambda x: ' '.join(x))

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
vocab = vectorizer.vocabulary_  # {단어: col_idx}
feature_names = vectorizer.get_feature_names_out()

print(f'TF-IDF 계산 완료: vocab {len(vocab):,}개 단어, {tfidf_matrix.shape[0]}개 문서')

In [ ]:
# hyperparameter
W_KNU    = 1.5
W_DESIGN = 1.2
W_EMOJI  = 0.35

def get_tfidf_score(doc_idx, word):
    """특정 문서에서 특정 단어의 TF-IDF 값 반환"""
    if word in vocab:
        return tfidf_matrix[doc_idx, vocab[word]]
    return 0.0

def compute_sentiment(doc_idx, tokens, emojis):
    """
    3계층 감성사전 + TF-IDF 가중 평균으로 polarity 계산
    """
    weighted_sum = 0.0
    weight_total = 0.0
    matched = {'knu': [], 'design': [], 'emoji': []}

    # 토큰 기반 (KNU + 디자인)
    for token in tokens:
        tfidf = get_tfidf_score(doc_idx, token)

        # KNU 사전
        if token in knu_lexicon:
            score = knu_lexicon[token]
            w = tfidf * W_KNU
            weighted_sum += score * w
            weight_total += abs(w)
            matched['knu'].append((token, round(score, 3), round(tfidf, 4)))

        # 디자인 사전
        if token in design_lexicon:
            score = design_lexicon[token]
            w = tfidf * W_DESIGN
            weighted_sum += score * w
            weight_total += abs(w)
            matched['design'].append((token, round(score, 3), round(tfidf, 4)))

    # 이모지 기반
    for em in emojis:
        if em in emoji_lexicon:
            score = emoji_lexicon[em]
            w = W_EMOJI
            weighted_sum += score * w
            weight_total += abs(w)
            matched['emoji'].append((em, round(score, 3)))

    if weight_total == 0:
        polarity = 0.0
    else:
        polarity = weighted_sum / weight_total
        polarity = max(-1.0, min(1.0, polarity))  # 클리핑

    return polarity, matched

In [ ]:
from tqdm.notebook import tqdm

polarities = []
matched_words = []

for i in tqdm(range(len(df)), desc='감성분석'):
    tokens = df['tokens_list'].iloc[i]
    emojis = df['emojis_list'].iloc[i]
    polarity, matched = compute_sentiment(i, tokens, emojis)
    polarities.append(round(polarity, 4))
    matched_words.append(matched)

df['polarity']      = polarities
df['matched_words'] = matched_words

# 감성 레이블
def label(p):
    if p >= 0.1:  return '긍정'
    elif p <= -0.1: return '부정'
    else:           return '중립'

df['sentiment_label'] = df['polarity'].apply(label)
print(df['sentiment_label'].value_counts())
print(f'\n평균 polarity: {df["polarity"].mean():.4f}')

In [ ]:
# 원문 + 결과 확인
show_cols = ['content', 'polarity', 'sentiment_label']
df[show_cols].sample(10, random_state=42)

In [ ]:
out = df.drop(columns=['tokens_list', 'emojis_list']).copy()
out['matched_words'] = out['matched_words'].astype(str)

out_path = '29cm_sentiment_result.csv'
out.to_csv(out_path, index=False, encoding='utf-8-sig')
files.download(out_path)